In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(lisi)
 
})

In [ ]:
detect_genes_fixed = function (cds, min_expr = 0) {
    assertthat::assert_that(methods::is(cds, "cell_data_set"))
    assertthat::assert_that(is.numeric(min_expr))
    mat_bin <- counts(cds) > min_expr
    rowData(cds)$num_cells_expressed <- Matrix::rowSums(mat_bin)
    colData(cds)$num_genes_expressed <- Matrix::colSums(mat_bin)
    cds
}

In [ ]:
doublet_scores_SIR_1 = read.table("../data/scrublet_doublet_scores_plate123.txt")
colnames(doublet_scores_SIR_1) = c("doublet_score")

doublet_scores_SIR_2 = read.table("../data/scrublet_doublet_scores_plate45.txt")
colnames(doublet_scores_SIR_2) = c("doublet_score")

In [ ]:
doublet_scores = rbind(doublet_scores_SIR_1, doublet_scores_SIR_2)

print(table(doublet_scores > 0.2))

## Supplementary Figure 2B
options(repr.plot.width=3, repr.plot.height=2.7, repr.plot.res = 300)
ggplot(doublet_scores, aes(x = doublet_score)) +
    geom_histogram(bins = 100) + 
    labs(x = "Scrublet score", y = "Count") + 
    scale_y_log10(labels = scales::comma) + 
    geom_vline(xintercept = 0.2, color="red") + 
    monocle3:::monocle_theme_opts() + 
    theme(axis.title = element_text(color="black", size = 12), 
          axis.text.x = element_text(color="black", size = 10), 
          axis.text.y = element_text(color = "black", size = 10),
          axis.ticks = element_line(color = "black"))

## Assign cell type labels

In [ ]:
cds_combined_poc = readRDS("cds_combined_VCS_JAX_poc.RDS")

In [ ]:
metadata = as.data.frame(colData(cds_combined_poc))
dim(metadata)

In [ ]:
JAX_main_trajectory = metadata[metadata$Data == "JAX",]$Main_trajectory
JAX_main_cell_type = metadata[metadata$Data == "JAX",]$Main_cell_type

In [ ]:
metadata[metadata$Data == "JAX",]$nn_majority_trajectory_pca = JAX_main_trajectory
metadata[metadata$Data == "JAX",]$nn_majority_trajectory_umap = JAX_main_trajectory

metadata[metadata$Data == "JAX",]$nn_majority_cell_type_pca = JAX_main_cell_type
metadata[metadata$Data == "JAX",]$nn_majority_cell_type_umap = JAX_main_cell_type

In [ ]:
trajectory_comp = metadata %>%
    group_by(Data, nn_majority_trajectory_pca) %>%
    summarise(n = n())%>%
    filter(n >= 30) %>%
    group_by(Data) %>% mutate(sum_n = sum(n), prop_n = n / sum_n) %>%
    group_by(nn_majority_trajectory_pca) %>%
    add_tally() %>% filter(nn == 2) %>% select(-nn) %>%
    select(-c(n, sum_n)) %>%
    tidyr::pivot_wider(., names_from = "Data", 
                       values_from = "prop_n") 

In [ ]:
trajectory_comp

In [ ]:
## Supplementary Figure 2C

options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
trajectory_comp %>%
    ggplot(., aes(x = JAX, y = VCS)) +
        geom_point(size = 1.2, alpha = 0.7) +
        geom_abline(slope = 1) + 
        labs(x = "Reference lineage proportion", y= "VCS lineage proportion") + 
        monocle3:::monocle_theme_opts() + 
        theme(axis.title = element_text(color="black", size = 16), 
              axis.text.x = element_text(color="black", size = 12), 
              axis.text.y = element_text(color = "black", size = 12),
              axis.ticks = element_line(color = "black"))

In [ ]:
cor(trajectory_comp$JAX, trajectory_comp$VCS, method = "pearson")

In [ ]:
length(sir_cells)

In [ ]:
trajectory_comp = metadata %>%
    group_by(Data, nn_majority_cell_type_pca) %>%
    summarise(n = n())%>%
    filter(n >= 30) %>%
    group_by(Data) %>% mutate(sum_n = sum(n), prop_n = n / sum_n) %>%
    group_by(nn_majority_cell_type_pca) %>%
    add_tally() %>% filter(nn == 2) %>% select(-nn) %>%
    select(-c(n, sum_n)) %>%
    tidyr::pivot_wider(., names_from = "Data", 
                       values_from = "prop_n") 

In [ ]:
## Supplementary Figure 2D

options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=300)
trajectory_comp %>%
    ggplot(., aes(x = JAX, y = VCS)) +
        geom_point(size = 1.2, alpha = 0.5) +
        geom_abline(slope = 1) + 
        labs(x = "Reference cell type proportion", y= "VCS cell type proportion") + 
        monocle3:::monocle_theme_opts() + 
        theme(axis.title = element_text(color="black", size = 16), 
              axis.text.x = element_text(color="black", size = 12), 
              axis.text.y = element_text(color = "black", size = 12),
              axis.ticks = element_line(color = "black"))

In [ ]:
cds_combined_poc

In [ ]:
subsample_cds = function(cds, n = 100000) {
    metadata = as.data.frame(colData(cds))
    
    metadata_JAX = metadata %>% filter(Data == "JAX") %>% sample_n(n)
    metadata_VCS = metadata %>% filter(Data == "VCS") %>% sample_n(n)

    cell_sampled = sample(c(metadata_JAX$Cell, metadata_VCS$Cell))
    
    cds_subsampled = cds[, cds$Cell %in% cell_sampled]
    
    return(cds_subsampled)
}


In [ ]:
cds_sampled = subsample_cds(cds_combined_poc, 10000)

In [ ]:
cds_sampled

In [ ]:
## Figure 1E
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=400)
g = plot_cells(cds_sampled, x = 1, y = 2,
           reduction_method = "UMAP", color_cells_by = "Data", 
           label_cell_groups = F,
           cell_size = 0.3) + 
        theme(legend.position = "none") + 
        scale_color_manual(values=c("#743282", "#f1a340"))

g$layers[[1]]$aes_params$alpha <- 0.2
g$layers[[2]]$aes_params$alpha <- NULL 
g + aes(alpha = Data) + scale_alpha_manual(values = c("JAX" = 0.5, "VCS" = 0.5))


## Load data

In [ ]:
metadata = as.data.frame(colData(cds_combined_poc))
JAX_main_trajectory = metadata[metadata$Data == "JAX",]$Main_trajectory
JAX_main_cell_type = metadata[metadata$Data == "JAX",]$Main_cell_type

metadata[metadata$Data == "JAX",]$nn_majority_trajectory_pca = JAX_main_trajectory
metadata[metadata$Data == "JAX",]$nn_majority_trajectory_umap = JAX_main_trajectory

metadata[metadata$Data == "JAX",]$nn_majority_cell_type_pca = JAX_main_cell_type
metadata[metadata$Data == "JAX",]$nn_majority_cell_type_umap = JAX_main_cell_type

In [ ]:
cds_combined_poc$nn_majority_trajectory_umap = metadata$nn_majority_trajectory_umap
cds_combined_poc$nn_majority_trajectory_pca = metadata$nn_majority_trajectory_pca
cds_combined_poc$nn_majority_cell_type_umap = metadata$nn_majority_cell_type_umap
cds_combined_poc$nn_majority_cell_type_pca = metadata$nn_majority_cell_type_pca

In [ ]:
pseudobulk_cds = function(cds_sub, sample_group = "nn_majority_trajectory_umap", 
                          cell_group = "nn_majority_cell_type_umap") {
    cf_ccs = new_cell_count_set(cds_sub,
                            sample_group = sample_group,
                            cell_group = cell_group)
    
    pb_cds = hooke:::pseudobulk_ccs_for_states(cf_ccs)
    
    return(pb_cds)
    
}

In [ ]:
pb_combined_poc = pseudobulk_cds(cds_combined_poc, sample_group = "Data", 
                                 cell_group = "nn_majority_trajectory_pca")

In [ ]:
top_subset_genes = readRDS("../data/highly_dispersed_genes_3000_E15-E15.5.RDS")

In [ ]:
trajectories = as.data.frame(colData(cds_combined_poc)) %>%
    distinct(nn_majority_trajectory_pca) %>% 
    filter(nn_majority_trajectory_pca != "Primitive_erythroid") %>%
    pull() 

In [ ]:
pb_metadata = as.data.frame(colData(pb_combined_poc)) %>%
                filter(num_cells_in_group >= 100) %>%
                group_by(cell_group) %>% add_tally() %>% filter(n == 2) %>% as.data.frame()
head(pb_metadata)

In [ ]:
JAX_ids = pb_metadata %>% filter(sample == "JAX") %>% arrange(cell_group) %>% pull(pseudobulk_id)
VCS_ids = pb_metadata %>% filter(sample == "VCS") %>% arrange(cell_group) %>% pull(pseudobulk_id)

In [ ]:
corr_mat = matrix(0, nrow = length(JAX_ids), ncol = length(VCS_ids))

for (i in 1:length(JAX_ids)) {
    JAX_id = JAX_ids[i]
    for (j in 1:length(VCS_ids)) {
        VCS_id = VCS_ids[j] 
        pb_sub = pb_combined_poc[top_subset_genes, c(JAX_id, VCS_id)]
        expr_df = normalized_counts(pb_sub)
        corr_mat[i,j] = cor(expr_df[,1], expr_df[,2], method = "pearson")
    }
}

In [ ]:
colnames(corr_mat) = pb_metadata %>% filter(sample == "JAX") %>% arrange(cell_group) %>% pull(cell_group)
rownames(corr_mat) = pb_metadata %>% filter(sample == "VCS") %>% arrange(cell_group) %>% pull(cell_group)
corr_mat

In [ ]:
draw_colnames_45 <- function (coln, gaps, ...) {
    coord <- pheatmap:::find_coordinates(length(coln), gaps)
    x     <- coord$coord - 0.5 * coord$size
    res   <- grid::textGrob(
      coln, x = x, y = unit(1, "npc") - unit(3,"bigpts"),
      vjust = 0.75, hjust = 1, rot = 45, gp = grid::gpar(...)
    )
    return(res)
}
assignInNamespace(
  x = "draw_colnames",
  value = "draw_colnames_45",
  ns = asNamespace("pheatmap")
)

In [ ]:
row_labels = rownames(corr_mat)
row_labels[3] = "Neuroectoderm_and_glia"
row_labels[11] = "Intermediate_neuronal_progenitors"

row_labels = gsub("_", " ", row_labels)

rownames(corr_mat) = row_labels
colnames(corr_mat) = row_labels

In [ ]:
ph = pheatmap(corr_mat, cluster_rows = F, cluster_cols = F, angle_col = 45)

In [ ]:
## Supplementary Figure 2E
options(repr.plot.width=7, repr.plot.height=6, repr.plot.res=300)
ph = pheatmap(corr_mat, cluster_rows = F, cluster_cols = F, angle_col = 45, silent=T)
ph$gtable$widths[[1]] <- unit(1.1, "inches")

grid.newpage()
grid.draw(ph$gtable)

In [ ]:
cds_combined_poc_all = readRDS("../data/cds_combined_VCS_JAX_poc_all_genes.RDS")

In [ ]:
pb_combined_poc_all = pseudobulk_cds(cds_combined_poc_all, sample_group = "Data", 
                                 cell_group = "nn_majority_trajectory_pca")

In [ ]:
marker_genes = read.table("../data/qiu_marker_genes.txt", header=T)
marker_genes = marker_genes %>%
  tidyr::separate_longer_delim(gene_short_name, delim = ",")

In [ ]:
marker_genes %>%
    filter(!gene_short_name %in% as.data.frame(rowData(pb_combined_poc_all))$gene_short_name)

In [ ]:
marker_genes = marker_genes %>%
    left_join(., as.data.frame(rowData(pb_combined_poc_all)) %>% select(gene_short_name, gene_ID)) %>%
    na.omit() %>%
    mutate(gene_ID = as.character(gene_ID))
    #count(trajectory)
marker_genes

In [ ]:
as.data.frame(rowData(pb_combined_poc_all)) %>%
    filter(gene_short_name %in% marker_genes$gene_short_name)

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
  gene_list = stringr::str_to_title(gene_list)
  cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
  aggregate_marker_expression = counts(cds_gene_list_norm)
  aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
  aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
  aggregate_marker_expression = log(aggregate_marker_expression + 1)
  colData(cds)[[name]] = aggregate_marker_expression
  return(cds)
}

In [ ]:
trajectories = sort(unique(pb_combined_poc_all$cell_group))
trajectories = trajectories[!trajectories %in% c("Primitive_erythroid")]

In [ ]:
for (traj in trajectories) {
    marker_genes_sub = marker_genes %>% filter(trajectory == traj) %>% pull(gene_short_name)
    
    pb_combined_poc_all = aggregate_gene_score(pb_combined_poc_all, marker_genes_sub, traj)
}

In [ ]:
pb_combined_poc_all

In [ ]:
options(repr.plot.width=6, repr.plot.height=5, repr.plot.res=300)
JAX_heatmap = as.data.frame(colData(pb_combined_poc_all)) %>%
    select(sample, cell_group, Adipocytes:White_blood_cells) %>%
    filter(sample == "JAX" & !cell_group %in% c("Primitive_erythroid")) %>%
    arrange(cell_group) %>% select(-sample) %>%
    remove_rownames() %>%
    column_to_rownames("cell_group") %>%
    mutate(across(where(is.numeric), ~ (. - min(., na.rm = TRUE)) / 
                                    (max(., na.rm = TRUE) - min(., na.rm = TRUE)))) %>%
    ungroup() 


row_labels = rownames(JAX_heatmap)
row_labels[3] = "Neuroectoderm_and_glia"
row_labels[11] = "Intermediate_neuronal_progenitors"

row_labels = gsub("_", " ", row_labels)

rownames(JAX_heatmap) = row_labels
colnames(JAX_heatmap) = row_labels

In [ ]:
## Supplementary Figure 2F

options(repr.plot.width=8, repr.plot.height=6.8, repr.plot.res=300)
ph = pheatmap(t(JAX_heatmap), cluster_rows = F, cluster_cols = F, angle_col = 45, silent=T)
ph$gtable$widths[[1]] <- unit(1.1, "inches")

grid.newpage()
grid.draw(ph$gtable)

In [ ]:
options(repr.plot.width=6, repr.plot.height=5, repr.plot.res=300)
VCS_heatmap = as.data.frame(colData(pb_combined_poc_all)) %>%
    select(sample, cell_group, Adipocytes:White_blood_cells) %>%
    filter(sample == "VCS" & !cell_group %in% c("Primitive_erythroid")) %>%
    arrange(cell_group) %>% select(-sample) %>%
    remove_rownames() %>%
    column_to_rownames("cell_group") %>%
    mutate(across(where(is.numeric), ~ (. - min(., na.rm = TRUE)) / 
                                    (max(., na.rm = TRUE) - min(., na.rm = TRUE)))) %>%
    ungroup()

In [ ]:
## Supplementary Figure 2G

rownames(VCS_heatmap) = row_labels
colnames(VCS_heatmap) = row_labels

options(repr.plot.width=8, repr.plot.height=6.8, repr.plot.res=300)
ph = pheatmap(t(VCS_heatmap), cluster_rows = F, cluster_cols = F, angle_col = 45, silent=T)
ph$gtable$widths[[1]] <- unit(1.1, "inches")

grid.newpage()
grid.draw(ph$gtable)

In [ ]:
pheatmap(t(VCS_heatmap), cluster_rows = F, cluster_cols = F, angle_col = 45)

## Supplementary Figure 3 Erythroids and hepatocytes

In [ ]:
metadata_poc = as.data.frame(colData(cds_combined_poc)) 
cds_combined_poc$UMAP_2d_1 = reducedDims(cds_combined_poc)[['UMAP']][,1]
cds_combined_poc$UMAP_2d_2 = reducedDims(cds_combined_poc)[['UMAP']][,2]

JAX_main_trajectory = metadata_poc[metadata_poc$Data == "JAX",]$Main_trajectory
JAX_main_cell_type = metadata_poc[metadata_poc$Data == "JAX",]$Main_cell_type

metadata_poc[metadata_poc$Data == "JAX",]$nn_majority_trajectory_pca = JAX_main_trajectory
metadata_poc[metadata_poc$Data == "JAX",]$nn_majority_trajectory_umap = JAX_main_trajectory

metadata_poc[metadata_poc$Data == "JAX",]$nn_majority_cell_type_pca = JAX_main_cell_type
metadata_poc[metadata_poc$Data == "JAX",]$nn_majority_cell_type_umap = JAX_main_cell_type

cds_combined_poc$nn_majority_trajectory_umap = metadata_poc$nn_majority_trajectory_umap
cds_combined_poc$nn_majority_trajectory_pca = metadata_poc$nn_majority_trajectory_pca
cds_combined_poc$nn_majority_cell_type_umap = metadata_poc$nn_majority_cell_type_umap
cds_combined_poc$nn_majority_cell_type_pca = metadata_poc$nn_majority_cell_type_pca

In [ ]:
cds_erythroids = cds_combined_poc[, 
                                cds_combined_poc$nn_majority_trajectory_pca == "Definitive_erythroid"]
cds_hepatocytes = cds_combined_poc[, 
                                cds_combined_poc$nn_majority_trajectory_pca == "Hepatocytes"]

In [ ]:
metadata_erythroids = as.data.frame(colData(cds_erythroids))
metadata_hepatocytes = as.data.frame(colData(cds_hepatocytes))

In [ ]:
metadata_erythroids 

In [ ]:
## Supplementary Figure 3A
options(repr.plot.width=6, repr.plot.height=5, repr.plot.res=300)
metadata_erythroids %>%
    filter(UMAP_2d_1 > 0.2 & UMAP_2d_2 > -2) %>%
    ggplot(., aes(x = UMAP_2d_1, y = UMAP_2d_2)) + 
            geom_point(aes(color = Data), size = 0.1, stroke = I(0.2)/2, alpha = 0.6) + 
            scale_color_manual(values=c("#743282", "#f1a340")) + 
            monocle3:::monocle_theme_opts()

In [ ]:
## Supplementary Figure 3B
options(repr.plot.width=8, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_erythroids,  
           genes = c("Gypa", "Tfrc", "Snca", "Slc4a1"),
           x = 1, y = 2, min_expr = 1.3,
           reduction_method = "UMAP", 
           color_cells_by = "Data", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 0.3, alpha = 1) +
        xlim(c(0.2, NA)) + ylim(c(-2, NA))


In [ ]:
## Supplementary Figure 3C
options(repr.plot.width=6, repr.plot.height=5, repr.plot.res=300)
metadata_hepatocytes %>%
    filter(UMAP_2d_2 < -15) %>%
    ggplot(., aes(x = UMAP_2d_1, y = UMAP_2d_2)) + 
            geom_point(aes(color = Data), size = 0.4, stroke = I(0.4)/2, alpha = 0.7) + 
            scale_color_manual(values=c("#743282", "#f1a340")) + 
            monocle3:::monocle_theme_opts()

In [ ]:
cds_hepatocytes = cds_hepatocytes[, cds_hepatocytes$UMAP_2d_2 < -15]

In [ ]:
cds_hepatocytes = detect_genes_fixed(cds_hepatocytes, 1)
hepatocytes_IDs = as.data.frame(rowData(cds_hepatocytes)) %>% filter(num_cells_expressed > 100) %>% pull(gene_ID)
cds_hepatocytes = cds_hepatocytes[as.character(hepatocytes_IDs), ]

In [ ]:
cds_hepatocytes

In [ ]:
hepatocytes_fit_models = fit_models(cds_hepatocytes, 
                             clean_model = T, verbose = T,
                             model_formula_str="~Data",
                             cores=1) 

In [ ]:
pseudo_coefs = coefficient_table(hepatocytes_fit_models)

In [ ]:
pseudo_coefs %>%
    select(-c(model, model_summary)) %>%
    filter(status == "OK"  & !grepl("Inter", term) & q_value < 0.01) %>%
    count(term)

In [ ]:
pseudo_coefs %>%
    filter(status == "OK"  & grepl("Data", term) & q_value < 1e-2 & abs(estimate) >= 0.5) %>%
    arrange(q_value) %>% dim()

In [ ]:
pseudo_coefs = pseudo_coefs %>%
    select(-c(model, model_summary)) %>%
    filter(status == "OK"  & grepl("Data", term))

In [ ]:
bulk_gene_list = read.csv("goncalves_b6_cast_degs.csv")
head(bulk_gene_list)

In [ ]:
pseudo_coefs = pseudo_coefs %>%
    left_join(., bulk_gene_list %>% select(gene_ID, log2FoldChange)) %>%
    mutate(in_goncalves = ifelse(is.na(log2FoldChange), F, T))

In [ ]:
cds_hepatocytes = detect_genes_fixed(cds_hepatocytes, 1)
hepatocytes_IDs = as.data.frame(rowData(cds_hepatocytes)) %>% filter(num_cells_expressed > 200) %>% pull(gene_ID)

In [ ]:
## Supplementary Figure 3D
options(repr.plot.width=8, repr.plot.height=5.5, repr.plot.res=300)

pseudo_coefs$significance <- "NS"
pseudo_coefs$significance[pseudo_coefs$q_value < 0.01 & pseudo_coefs$estimate > log(2)] <- "Up in VCS"
pseudo_coefs$significance[pseudo_coefs$q_value < 0.01 & pseudo_coefs$estimate < -log(2)] <- "Down in VCS"
pseudo_coefs$significance[pseudo_coefs$in_goncalves == T] <- "Sig. in CAST"

genes_to_highlight = c("Afp", "Alb", "Trf", "Fga", "Fgb", "Fgg",
                       "Yap1", "Trim71",
                       "Sik3", "Gab1", "Mcu", "Cyp2c67")

pseudo_coefs %>%
    filter(gene_id %in% hepatocytes_IDs) %>%
    filter(chr != "chrX") %>%
    ggplot(aes(x = estimate, y = -log10(q_value), color = significance)) +
        geom_point(alpha = 0.7, size = 1.2) +
        
        geom_point(data = . %>% filter(gene_short_name %in% genes_to_highlight), 
                   color = "black", size = 3, shape = 1, stroke = 1.2) +
        
        geom_hline(yintercept = -log10(0.01), linetype = "dashed") +
        geom_vline(xintercept = c(-log(2), log(2)), linetype = "dashed") +

        ggrepel::geom_label_repel(data = . %>% 
                                 filter(gene_short_name %in% genes_to_highlight),
                        aes(label = gene_short_name),
                        color = "black",    
                        fill = "white",
                        size = 5,
                        fontface = "italic",
                        box.padding = 0.8,
                        point.padding = 0.3, 
                        min.segment.length = 0, 
                        show.legend = FALSE) + 
                        
        scale_color_manual(values = c("Up in VCS" = "red", "Down in VCS" = "skyblue", 
                                      "NS" = "grey", "Sig. in CAST" = "black")) +
        xlim(c(-5, 5)) + 
        theme_minimal() +
        labs(
           x = "Effect size (Quasi-Poisson)",
           y = "-log10 adjusted p-value") +
        theme(axis.title = element_text(color="black", size = 16), 
              axis.text.x = element_text(color="black", size = 16), 
              axis.text.y = element_text(color = "black", size = 16))

In [ ]:
pseudo_coefs %>%
    filter(q_value < 1e-2 & abs(estimate) >= log(2)) %>%
    filter(chr != "chrX") %>%
    filter(gene_id %in% hepatocytes_IDs) %>% 
    mutate(direction = ifelse(estimate > log(2), "Up", "Down")) %>%
    count(direction) %>%
    mutate(prop = n / sum(n) * 100,
           label = paste0(n, " (", round(prop, 1), "%)"))

In [ ]:
pseudo_coefs_prop = pseudo_coefs %>%
    filter(q_value < 1e-2 & abs(estimate) >= log(2)) %>%
    filter(chr != "chrX") %>%
    filter(gene_id %in% hepatocytes_IDs) %>% 
    mutate(direction = ifelse(estimate > log(2), "Up", "Down")) %>%
    count(direction) %>%
    mutate(prop = n / sum(n) * 100,
           label = paste0(n, " (", round(prop, 1), "%)"))

## Supplementary Figure 3E
options(repr.plot.width=3.5, repr.plot.height=5, repr.plot.res=300)
ggplot(pseudo_coefs_prop, aes(x = "", y = n, fill = direction)) +
  geom_bar(stat = "identity", width = 0.4, color = "black") +
  scale_fill_manual(values = c("skyblue", "red")) +
  theme_classic() +
  labs(
    x = "", 
    y = "Count", 
    fill = "Status",
  ) +
  theme(
    axis.text.x = element_text(size = 16, color = "black"),
    axis.text.y = element_text(size = 16, color = "black"),
    axis.title.y = element_text(size = 16, color = "black"),
    legend.position = "none",
    axis.ticks.x = element_blank()
  )

In [ ]:
pseudo_coefs %>%
    filter(q_value < 1e-2) %>%
    filter(chr != "chrX") %>%
    filter(gene_id %in% hepatocytes_IDs) %>% 
    count(in_goncalves) %>%
    mutate(prop = n / sum(n) * 100,
            label = paste0(n, " (", round(prop, 1), "%)")) 

In [ ]:
pseudo_coefs_prop = pseudo_coefs %>%
        filter(q_value < 1e-2 & abs(estimate) >= log(2)) %>%
        filter(chr != "chrX") %>%
        filter(gene_id %in% hepatocytes_IDs) %>% 
        count(in_goncalves) %>%
        mutate(prop = n / sum(n) * 100,
               label = paste0(n, " (", round(prop, 1), "%)")
               in_goncalves = ifelse(in_goncalves, "DE_in_CAST", "Not_De"))

## Supplementary Figure 3E
options(repr.plot.width=3.5, repr.plot.height=5, repr.plot.res=300)
ggplot(pseudo_coefs_prop, aes(x = "", y = n, fill = in_goncalves)) +
  geom_bar(stat = "identity", width = 0.4, color = "black") +
  scale_fill_manual(values = c("black", "grey")) +
  theme_classic() +
  labs(
    x = "", 
    y = "Count", 
    fill = "Status",
  ) +
  theme(
    axis.text.x = element_text(size = 16, color = "black"),
    axis.text.y = element_text(size = 16, color = "black"),
    axis.title.y = element_text(size = 16, color = "black"),
    legend.position = "none",
    axis.ticks.x = element_blank()
  )

In [ ]:
pseudo_coefs %>%
    filter(q_value < 1e-2 & abs(estimate) >= log(2)) %>%
    filter(chr != "chrX") %>%
    filter(gene_id %in% hepatocytes_IDs) %>% 
    mutate(direction = ifelse(estimate > log(2), "Up", "Down")) %>%
    filter(direction == "Up") %>%
    arrange(q_value) %>% slice(1:50)

In [ ]:
pseudo_coefs %>%
    filter(q_value < 1e-2 & abs(estimate) >= log(2)) %>%
    filter(chr != "chrX") %>%
    filter(gene_id %in% hepatocytes_IDs) %>% 
    mutate(direction = ifelse(estimate > log(2), "Up", "Down")) %>%
    filter(direction == "Down") %>%
    arrange(q_value) %>% slice(1:50) 

In [ ]:
## Supplementary Figure 3F
options(repr.plot.width=13, repr.plot.height=6, repr.plot.res=300)
plot_cells(cds_hepatocytes,  scale_to_range = F,
           genes = c("Afp", "Alb"),
           x = 1, y = 2, min_expr = 5,
           reduction_method = "UMAP", 
           color_cells_by = "Data", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 1.3, alpha = 1) +
        theme(aspect.ratio = 1) + 
        ylim(c(NA, -15)) +
        guides(color = guide_colorbar(title = "log10\nexpression")) +
        theme(strip.text = element_text(size = 28),
            legend.title = element_text(size = 14),
            legend.text = element_text(size = 14) 
        )

In [ ]:
## Supplementary Figure 3G
options(repr.plot.width=12, repr.plot.height=6, repr.plot.res=300)
plot_cells(cds_hepatocytes,  scale_to_range = F,
           genes = c("Yap1", "Trim71"),
           x = 1, y = 2, min_expr = 0.3,
           reduction_method = "UMAP", 
           color_cells_by = "Data", 
           group_label_size = 3,
           label_cell_groups = F,
           cell_size = 1.4, alpha = 1) +
        theme(aspect.ratio = 1) + 
        ylim(c(NA, -15)) +
        guides(color = guide_colorbar(title = "log10\nexpression")) +
        theme(strip.text = element_text(size = 28),
            legend.title = element_text(size = 14),
            legend.text = element_text(size = 14) 
        )
    